# Updating the book

The answers to Section 2 of the sample questions, `documentation/tex/questions`, run.
The questions concern the update of the aggregated book by an incoming order, Propositions
1.1.10 and 1.1.11 of the lecture notes, its implementation in Section 1.2.3, and the
placement of orders under Assumption 1.4.1.

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

The book of most questions, as two maps from price to size, and a reading of a book as
one signed map (bids positive, asks negative), so that two books can be compared.

In [ ]:
BIDS = {1000: 100, 999: 200, 998: 150}
ASKS = {1002: 120, 1003: 180}


def baseline():
    return AggregateBook.from_levels(BIDS, ASKS)


def state(book):
    signed = {price: size for price, size in book.levels_map(BUY).items()}
    signed.update({price: -size for price, size in book.levels_map(SELL).items()})
    return dict(sorted(signed.items()))

## `question.limitBuyRestsInsideTheSpread`: the next state, and three quantities derived from it

In [ ]:
def run():
    book = AggregateBook.from_levels(
        {1000: 300, 999: 100}, {1001: 50, 1002: 100, 1004: 200}
    )
    book.apply(limit_order(1.0, 200, 1003, BUY), record=False)
    print(book.spread, book.mid_price,
          round(book.queue_imbalance(GridDepth(1)), 3))


printed = outcome(run)
print(printed)
verify("question.limitBuyRestsInsideTheSpread", "C", [
    printed == "2 1003.0 -0.6",
    printed == "1 1003.5 0.6",
    printed == "1 1003.5 -0.6",
    printed == "4 1002.0 0.2",
    printed == "4 1002.0 0.333",
])

## `question.precomputedMarketPartOnAWalkingSell`: the one code path against a market part computed first

In [ ]:
def submit(book, message):
    remaining, d, p = message.size, message.direction, message.price
    fills = []
    while remaining > 0:
        best = book.best_price(-d)
        if best is None or best * d > p * d:
            break
        resting = book.size_at(-d, best)
        traded = min(remaining, resting)
        remaining -= traded
        book.set_size(-d, best, resting - traded)
        fills.append((best, traded))
    last = fills[-1][0] if fills else None
    rest = book.resting_price(p, last)
    if remaining > 0 and rest is not None:
        book.set_size(d, rest, book.size_at(d, rest) + remaining)
    return fills

submit_a = submit

In [ ]:
def submit(book, message):
    q, d, p = message.size, message.direction, message.price
    side = book.levels_map(-d)
    eligible = sum(s for price, s in side.items() if price * d <= p * d)
    q_m = min(q, eligible)
    fills, remaining = [], q_m
    while remaining > 0:
        best = book.best_price(-d)
        resting = book.size_at(-d, best)
        traded = min(remaining, resting)
        remaining -= traded
        book.set_size(-d, best, resting - traded)
        fills.append((best, traded))
    if q > q_m:
        book.set_size(d, p, book.size_at(d, p) + q - q_m)
    return fills

submit_b = submit

Alternative A is the update of the package: on every message of the catalogue of worked
transitions, and on its mirror image, it leaves the book that `AggregateBook.apply` leaves.

In [ ]:
from unito26.lob.messages import MessageType
from unito26.lob.worked_examples import CATALOGUE, reflect, to_sides

SUBMISSIONS = [
    example
    for example in CATALOGUE + tuple(reflect(example, 1001) for example in CATALOGUE)
    if example.message.kind is MessageType.SUBMIT
]
for example in SUBMISSIONS:
    ours = AggregateBook.from_levels(*to_sides(example.before))
    package = AggregateBook.from_levels(*to_sides(example.before))
    fills = submit_a(ours, example.message)
    result = package.apply(example.message, record=True)
    assert fills == [(fill.price, fill.size) for fill in result.fills], example.name
    assert state(ours) == state(package) == dict(sorted(example.after.items())), example.name
print(f"Alternative A agrees with the package on {len(SUBMISSIONS)} submissions")

In [ ]:
def run(submit):
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200, 998: 150}, {1002: 120, 1003: 180}
    )
    fills = submit(book, limit_order(1.0, 400, 999, SELL))
    print(fills)
    print(book.best_bid_price, book.best_ask_price, book.best_ask_size)


a, b = outcome(run, submit_a), outcome(run, submit_b)
print("A:", a.split("\n"), "   B:", b.split("\n"))
verify("question.precomputedMarketPartOnAWalkingSell", "A", [
    a == b == "[(1000, 100), (999, 200)]\n998 999 100",
    a == "[(1000, 100), (999, 200)]\n998 999 100"
    and b == "[(1000, 100), (999, 200), (998, 100)]\n998 1002 120",
    a == b == "[(999, 300)]\n998 999 100",
    a == "[(1000, 100), (999, 200)]\n998 999 100"
    and b == "[(1000, 100), (999, 200), (998, 50)]\n998 999 50",
])

## `question.precomputedMarketPartRestsAtTheSentinel`: an oversized market sell, then a limit buy

In [ ]:
def run(submit):
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200, 998: 150}, {1002: 120, 1003: 180}
    )
    submit(book, market_order(1.0, 1000, SELL))
    fills = submit(book, limit_order(2.0, 50, 997, BUY))
    print(fills, book.best_bid_price, book.best_ask_price)


a, b = outcome(run, submit_a), outcome(run, submit_b)
print("A:", a, "   B:", b)
verify("question.precomputedMarketPartRestsAtTheSentinel", "D", [
    a == b == "[] 997 998",
    a == b == "[] 997 1002",
    a == "[] 997 1002" and b == "[(0, 50)] None 0",
    a == "[] 997 998" and b == "[(0, 50)] None 0",
])

## `question.eligibleBidsWalkedFromTheLowest`: the eligible levels, sorted

In [ ]:
def submit(book, message):
    remaining, d, p = message.size, message.direction, message.price
    side = book.levels_map(-d)
    eligible = [price for price in side if price * d <= p * d]
    fills = []
    for price in sorted(eligible):
        if remaining == 0:
            break
        traded = min(remaining, side[price])
        remaining -= traded
        book.set_size(-d, price, side[price] - traded)
        fills.append((price, traded))
    if remaining > 0:
        book.set_size(d, p, book.size_at(d, p) + remaining)
    return fills

submit_sorted = submit

In [ ]:
def run(submit):
    for q in (400, 250):
        book = AggregateBook.from_levels(
            {1000: 100, 999: 200, 998: 150}, {1002: 120, 1003: 180}
        )
        submit(book, limit_order(1.0, q, 999, SELL))
        print(q, book.best_bid_price, book.best_bid_size, book.best_ask_price)


a, variant = outcome(run, submit_a), outcome(run, submit_sorted)
print("A:", a.split("\n"), "   variant:", variant.split("\n"))
assert a == "400 998 150 999\n250 999 50 1002", "the stem states what A prints"
verify("question.eligibleBidsWalkedFromTheLowest", "B", [
    variant == "400 998 150 999\n250 999 50 1002",
    variant == "400 998 150 999\n250 1000 50 1002",
    variant == "400 1000 100 999\n250 1000 100 1002",
    variant == "RuntimeError",
])

## `question.fillsPricedAtTheIncomingOrder`: the price a fill is recorded at

In [ ]:
def submit(book, message):
    remaining, d, p = message.size, message.direction, message.price
    fills = []
    while remaining > 0:
        best = book.best_price(-d)
        if best is None or best * d > p * d:
            break
        resting = book.size_at(-d, best)
        traded = min(remaining, resting)
        remaining -= traded
        book.set_size(-d, best, resting - traded)
        fills.append((p, traded))
    last = fills[-1][0] if fills else None
    rest = book.resting_price(p, last)
    if remaining > 0 and rest is not None:
        book.set_size(d, rest, book.size_at(d, rest) + remaining)
    return fills

submit_at_p = submit

In [ ]:
def run(submit):
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200, 998: 150}, {1002: 120, 1003: 180}
    )
    volume = value = 0
    for message in [limit_order(1.0, 250, 999, SELL),
                    market_order(2.0, 100, SELL)]:
        for price, size in submit(book, message):
            volume += size
            value += price * size
    print(book.best_bid_price, book.best_bid_size, volume, value)


a, variant = outcome(run, submit_a), outcome(run, submit_at_p)
print("A:", a, "   variant:", variant)
verify("question.fillsPricedAtTheIncomingOrder", "A", [
    a == "998 100 350 349700" and variant == "998 100 350 249750",
    a == variant == "998 100 350 349700",
    a == "998 100 350 349700" and variant == "998 100 350 349550",
    a == "999 50 250 249850" and variant == "999 50 250 249750",
])

## `question.priceTestWithoutTheDirection`: a price test written for one side

Each message is applied to its own copy of the book, once by Alternative A and once by the
variant, and the two books are compared.

In [ ]:
def submit(book, message):
    remaining, d, p = message.size, message.direction, message.price
    fills = []
    while remaining > 0:
        best = book.best_price(-d)
        if best is None or best > p:
            break
        resting = book.size_at(-d, best)
        traded = min(remaining, resting)
        remaining -= traded
        book.set_size(-d, best, resting - traded)
        fills.append((best, traded))
    last = fills[-1][0] if fills else None
    rest = book.resting_price(p, last)
    if remaining > 0 and rest is not None:
        book.set_size(d, rest, book.size_at(d, rest) + remaining)
    return fills

submit_directionless = submit

In [ ]:
messages = [
    limit_order(1.0, 200, 1002, BUY),   # (i)
    limit_order(1.0, 50, 1000, SELL),   # (ii)
    market_order(1.0, 50, SELL),        # (iii)
    limit_order(1.0, 50, 1001, SELL),   # (iv)
]

In [ ]:
agree = []
for numeral, message in zip(("i", "ii", "iii", "iv"), messages):
    ours, theirs = baseline(), baseline()
    submit_a(ours, message)
    submit_directionless(theirs, message)
    agree.append(state(ours) == state(theirs))
    print(f"({numeral}) A: {state(ours)}\n      variant: {state(theirs)}")
print(agree)
verify("question.priceTestWithoutTheDirection", "C", [
    agree == [True, False, False, False],
    agree == [True, True, True, False],
    agree == [True, True, False, False],
    agree == [True, True, False, True],
    agree == [True, True, True, True],
])

## `question.splittingAMarketOrderLosesItsRemainder`: an order replaced by its two parts

In [ ]:
def split(book, message):
    d, p = message.direction, message.price
    side = book.levels_map(-d)
    eligible = sum(s for price, s in side.items() if price * d <= p * d)
    q_m = min(message.size, eligible)
    return [
        market_order(message.time, q_m, d),
        limit_order(message.time, message.size - q_m, p, d),
    ]

In [ ]:
def run():
    bids = {1000: 100, 999: 200, 998: 150}
    asks = {1002: 120, 1003: 180}
    for message in [limit_order(1.0, 400, 999, SELL),
                    market_order(1.0, 1000, SELL),
                    market_order(1.0, 150, BUY)]:
        whole = AggregateBook.from_levels(bids, asks)
        parts = AggregateBook.from_levels(bids, asks)
        whole.apply(message, record=False)
        for part in split(parts, message):
            parts.apply(part, record=False)
        print(whole.bids == parts.bids and whole.asks == parts.asks)


lines = outcome(run).split("\n")
print(lines)
verify("question.splittingAMarketOrderLosesItsRemainder", "B", [
    lines == ["True", "True", "True"],
    lines == ["True", "False", "True"],
    lines == ["False", "False", "True"],
    lines == ["True", "False", "ValueError"] or lines == ["ValueError"],
])

## `question.setSizeKeepingEmptyLevelsNeverReturns`: a write that keeps a price of size zero

The second call of `apply` does not return.
To show it, `set_size` is watched while the snippet runs: after many calls the guard stops
the run, and the calls it saw last are all the same write, of size zero at 1002, on a book
that each of them leaves as it found it.
Besides the book and `remaining`, the loop of `submit` reads only what no pass changes,
so a pass that changes neither is followed by the same pass for ever.

In [ ]:
class ZeroKeepingBook(AggregateBook):
    def set_size(self, direction, price, size):
        self.levels_map(direction)[price] = size

In [ ]:
def run():
    for q in (120, 150):
        book = ZeroKeepingBook.from_levels(
            {1000: 100, 999: 200}, {1002: 120, 1003: 180}
        )
        book.apply(market_order(1.0, q, BUY), record=False)
        print(book.best_ask_price, book.best_ask_size, book.spread)


class Stuck(Exception):
    pass


written = []


class Watched(ZeroKeepingBook):
    def set_size(self, direction, price, size):
        written.append((direction, price, size, state(self)))
        if len(written) > 100_000:
            raise Stuck
        super().set_size(direction, price, size)


def watching(book_class):
    """What `run` prints with `book_class` in the place of ZeroKeepingBook, and what it raises."""
    global ZeroKeepingBook
    verbatim, ZeroKeepingBook = ZeroKeepingBook, book_class
    written.clear()
    printed = io.StringIO()
    raised = None
    try:
        with contextlib.redirect_stdout(printed):
            run()
    except Exception as error:
        raised = type(error).__name__
    finally:
        ZeroKeepingBook = verbatim
    return printed.getvalue().strip().split("\n"), raised


lines, raised = watching(Watched)
last = written[-1000:]
fixed_point = (
    len({repr(write) for write in last}) == 1
    and last[0][:3] == (SELL, 1002, 0)
    and last[0][3][1002] == 0
)
print(lines, raised, "  the last writes are one write of zero at 1002:", fixed_point)
print("the package's set_size:", watching(AggregateBook))
verify("question.setSizeKeepingEmptyLevelsNeverReturns", "D", [
    lines == ["1003 180 3", "1003 150 3"] and raised is None,
    lines == ["1002 0 2", "1002 0 2"] and raised is None,
    lines == ["1002 0 2"] and raised == "AssertionError",
    lines == ["1002 0 2"] and raised == "Stuck" and fixed_point,
])

## `question.withdrawalBeyondTheRestingSize`: a lenient book and a strict one

In [ ]:
def run(strict):
    book = AggregateBook.from_levels(
        {1000: 100, 999: 200}, {1002: 120, 1003: 180}, strict
    )
    book.apply(market_order(1.0, 30, SELL), record=False)
    book.apply(withdrawal(2.0, 100, 1000, BUY), record=False)
    print(book.best_bid_price, book.spread, book.mid_price)


lenient, strict = outcome(run, False), outcome(run, True)
print("strict=False:", lenient, "   strict=True:", strict)
verify("question.withdrawalBeyondTheRestingSize", "A", [
    lenient == "999 3 1000.5" and strict == "ValueError",
    lenient == "1000 2 1001.0" and strict == "ValueError",
    lenient == strict == "999 3 1000.5",
    lenient == strict == "ValueError",
])

## `question.sweepRescanningTheBestPrice`: a sweep of the whole side, timed

A market sell of $\ell$ shares against $\ell$ bid levels of one share each.
The book is built afresh, outside the timing, before each run.
The six cases are timed in turn, seven rounds over, and each keeps its fastest run,
so that a slow phase of the machine does not fall on one case alone.
The choices are read off the growth of each time from $\ell = 10^3$ to $\ell = 10^4$.

In [ ]:
def submit(book, message):
    remaining, d, p = message.size, message.direction, message.price
    side = book.levels_map(-d)
    eligible = sorted(
        (price for price in side if price * d <= p * d),
        key=lambda price: price * d,
    )
    fills = []
    for best in eligible:
        if remaining == 0:
            break
        resting = side[best]
        traded = min(remaining, resting)
        remaining -= traded
        book.set_size(-d, best, resting - traded)
        fills.append((best, traded))
    last = fills[-1][0] if fills else None
    rest = book.resting_price(p, last)
    if remaining > 0 and rest is not None:
        book.set_size(d, rest, book.size_at(d, rest) + remaining)
    return fills

submit_presorted = submit

The two leave the same book, on the catalogue of worked transitions and on the sweep.

In [ ]:
for example in SUBMISSIONS:
    ours = AggregateBook.from_levels(*to_sides(example.before))
    theirs = AggregateBook.from_levels(*to_sides(example.before))
    assert submit_a(ours, example.message) == submit_presorted(theirs, example.message)
    assert state(ours) == state(theirs), example.name


def sweep_book(depth):
    return AggregateBook.from_levels(
        {100_000 - k: 1 for k in range(depth)}, {100_001: 1}
    )


ours, theirs = sweep_book(2_000), sweep_book(2_000)
assert submit_a(ours, market_order(1.0, 2_000, SELL)) == submit_presorted(
    theirs, market_order(1.0, 2_000, SELL))
assert state(ours) == state(theirs) == {100_001: -1}

In [ ]:
def seconds(submit, depth):
    timer = timeit.Timer(
        "submit(book, message)",
        setup="book = sweep_book(depth)",
        globals={
            "submit": submit, "sweep_book": sweep_book, "depth": depth,
            "message": market_order(1.0, depth, SELL),
        },
    )
    return timer.timeit(number=1)


depths = (1_000, 3_000, 10_000)
cases = [(submit, depth) for depth in depths for submit in (submit_a, submit_presorted)]
rounds = [[seconds(submit, depth) for submit, depth in cases] for _ in range(7)]
fastest = dict(zip(cases, map(min, zip(*rounds))))
timings = {
    depth: (fastest[submit_a, depth], fastest[submit_presorted, depth]) for depth in depths
}
for depth in depths:
    print(f"l = {depth:>6}:  A {timings[depth][0] * 1e3:9.2f} ms   "
          f"B {timings[depth][1] * 1e3:7.2f} ms")

growth_a = timings[10_000][0] / timings[1_000][0]
growth_b = timings[10_000][1] / timings[1_000][1]
a_over_b = timings[10_000][0] / timings[10_000][1]
print(f"growth of A {growth_a:.0f}x, growth of B {growth_b:.1f}x, A over B at 10^4 {a_over_b:.0f}x")
verify("question.sweepRescanningTheBestPrice", "C", [
    a_over_b < 1,
    growth_a < 20 and growth_b < 20 and a_over_b > 1,
    growth_a > 50 and growth_b < 20 and a_over_b > 5,
    growth_a > 50 and growth_b > 50,
])

## `question.idealisedLimitBuyAtAFullTouch`: three limit buys under the idealised placement

`flow` reads nothing of its Hawkes process in `_place`; any six-type process serves.

In [ ]:
from unito26.lob.idealised import IdealisedFlow, UniformDepth
from unito26.lob.simulate import EventType

flow = IdealisedFlow(
    ExponentialHawkes(config.resilient_order_flow_params(), rng=0), UniformDepth(5)
)

In [ ]:
def run():
    book = AggregateBook.from_levels(
        {1000: 4, 999: 5, 998: 5}, {1001: 1, 1002: 5, 1003: 5}
    )
    prices = []
    for time in (1.0, 2.0, 3.0):
        message = flow._place(book, EventType.LIMIT_BUY, time)
        book.apply(message, record=False)
        prices.append(message.price)
    print(prices, book.best_bid_price, book.best_bid_size,
          book.best_ask_price, book.best_ask_size)


printed = outcome(run)
print(printed)
verify("question.idealisedLimitBuyAtAFullTouch", "B", [
    printed == "[1000, 1000, 1001] 1000 6 1002 5",
    printed == "[1000, 1001, 1001] 1001 1 1002 5",
    printed == "[1000, 1001, 1002] 1000 5 1002 4",
    printed == "AssumptionBreach",
])